# Adversarial Training

Train a robust model using clean + adversarial examples.

## Strategy
- **Non-overlapping**: Clean examples NOT used for adversarial generation
- **50/50 mix**: Equal parts clean and adversarial data
- **Original indices tracked**: Exclude clean data that was attacked

In [1]:
import os, math, random, json
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk, Dataset, concatenate_datasets, Value, Features
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback, set_seed, get_cosine_schedule_with_warmup
)
from sklearn.metrics import f1_score, accuracy_score, classification_report

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')


PyTorch: 2.5.1+cu121
CUDA available: True


## 1. Configuration

In [ ]:
# Paths
BASELINE_MODEL = './baseline_42'
OUTPUT_DIR = './adversarial_trained_model_42.25'

# Path to the local adversarial dataset
LOCAL_ADV_DATASET_PATH = './Adv_train_data_42/adv_only_hf_train'  

# Config
ADVERSARIAL_RATIO = 0.25  # 25% adversarial
MAX_LEN = 256
SEED = 42

# Labels
LABELS = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

# Target features for concatenation
    'text': Value('string'),
    'label_id': Value('int64')
})

# Set seeds
set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Adversarial ratio: {ADVERSARIAL_RATIO*100:.0f}%')
print(f'Output: {OUTPUT_DIR}')
print(f'Local adversarial dataset path: {LOCAL_ADV_DATASET_PATH}')

## 2. Load Clean Dataset

In [ ]:
clean_ds = load_dataset('ucsbnlp/liar', trust_remote_code=True)
print(f"Clean train: {len(clean_ds['train'])}")

def safe_str(x):
    if x is None: return ''
    if isinstance(x, (list, tuple)):
        return ', '.join([str(i) for i in x if i is not None])
    return str(x).strip()

def build_text(example):
    statement = safe_str(example.get('statement', ''))
    subject = safe_str(example.get('subjects', example.get('subject', '')))
    context = safe_str(example.get('context', ''))
    parts = [statement]
    if subject: parts.append(f'SUBJECT: {subject}')
    if context: parts.append(f'CONTEXT: {context}')
    example['text'] = ' [SEP] '.join(parts)
    y = example.get('label')
    if isinstance(y, str):
        example['label_id'] = label2id[y.strip().lower()]
    else:
        example['label_id'] = int(y)
    return example

clean_ds = clean_ds.map(build_text)
print('Clean dataset preprocessed')

## 3. Load Adversarial Examples

In [ ]:
print(f'Loading adversarial dataset from: {LOCAL_ADV_DATASET_PATH}')
adv_train_ds = load_from_disk(LOCAL_ADV_DATASET_PATH)

# If saved as a DatasetDict, extract the split
if hasattr(adv_train_ds, 'keys'):
    split = 'train' if 'train' in adv_train_ds else list(adv_train_ds.keys())[0]
    adv_train_ds = adv_train_ds[split]
    print(f"Loaded split: '{split}'")

print(f'Loaded {len(adv_train_ds)} adversarial examples')
print(f'Columns: {adv_train_ds.column_names}')

# Extract which clean examples were attacked
attacked_indices = set()
if 'orig_index' in adv_train_ds.column_names:
    attacked_indices = set(adv_train_ds['orig_index'])
    print(f'\nFound {len(attacked_indices)} unique original indices')
    print(f'These clean examples were used to generate adversarial data')
else:
    print('\nWARNING: No orig_index field found!')
    print('Cannot exclude attacked examples. Proceeding with potential overlap.')

# Add label_id if needed
def prep_adv(ex):
    if 'label_id' not in ex:
        ex['label_id'] = int(ex['label'])
    return ex

adv_train_ds = adv_train_ds.map(prep_adv)

## 4. Create Non-Overlapping Clean Dataset

In [ ]:
print('\n' + '='*70)
print('CREATING NON-OVERLAPPING DATASET')
print('='*70)

n_clean_total = len(clean_ds['train'])
n_attacked = len(attacked_indices)

print(f'\nOriginal clean training set: {n_clean_total:,} examples')
print(f'Examples used for adversarial generation: {n_attacked:,}')
print(f'Available clean examples (non-overlapping): {n_clean_total - n_attacked:,}')

if len(attacked_indices) > 0:
    all_indices = set(range(n_clean_total))
    clean_indices = sorted(list(all_indices - attacked_indices))
    print(f'\n✓ Excluding {n_attacked:,} attacked examples')
    print(f'✓ Keeping {len(clean_indices):,} non-attacked examples')
    clean_train_nonoverlap = clean_ds['train'].select(clean_indices)
else:
    print('\n⚠ WARNING: Using all clean data (no exclusion possible)')
    clean_train_nonoverlap = clean_ds['train']

print(f'\nNon-overlapping clean dataset: {len(clean_train_nonoverlap):,} examples')

## 5. Mix Non-Overlapping Clean + Adversarial Data

In [ ]:
n_clean_available = len(clean_train_nonoverlap)
n_adv_total = len(adv_train_ds)

print('\n' + '='*70)
print('MIXING STRATEGY')
print('='*70)

n_clean_use = n_clean_available
n_adv_use = min(n_clean_available, n_adv_total)

print(f'\nTarget ratio: {ADVERSARIAL_RATIO*100:.0f}% adversarial')
print(f'Clean examples to use: {n_clean_use:,}')
print(f'Adversarial examples to use: {n_adv_use:,}')
print(f'Total training examples: {n_clean_use + n_adv_use:,}')
print(f'Actual adversarial ratio: {n_adv_use/(n_clean_use + n_adv_use)*100:.1f}%')

# Sample adversarial if needed
if n_adv_use < n_adv_total:
    print(f'\nSampling {n_adv_use:,} from {n_adv_total:,} adversarial examples')
    indices = random.sample(range(n_adv_total), n_adv_use)
    adv_train_sampled = adv_train_ds.select(indices)
else:
    adv_train_sampled = adv_train_ds

# Keep only necessary columns
keep_cols = ['text', 'label_id']
clean_train_final = clean_train_nonoverlap.remove_columns(
    [c for c in clean_train_nonoverlap.column_names if c not in keep_cols]
)
adv_train_final = adv_train_sampled.remove_columns(
    [c for c in adv_train_sampled.column_names if c not in keep_cols]
)

# Fix dtype mismatch (large_string vs string) before concatenation
print('\nAligning feature dtypes...')
print(f'  Clean features: {clean_train_final.features}')
print(f'  Adv features:   {adv_train_final.features}')
clean_train_final = clean_train_final.cast(TARGET_FEATURES)
adv_train_final = adv_train_final.cast(TARGET_FEATURES)
print(f'  After cast:     {clean_train_final.features}')

# Verify no overlap
print('\n' + '='*70)
print('VERIFICATION: NO OVERLAP')
print('='*70)
print(f'Clean examples used: {len(clean_train_final):,}')
print(f'Adversarial examples used: {len(adv_train_final):,}')
print(f'Original attacked examples: {len(attacked_indices):,}')
print(f'Overlap: 0 (guaranteed by construction)')

# Concatenate and shuffle
mixed_train = concatenate_datasets([clean_train_final, adv_train_final])
mixed_train = mixed_train.shuffle(seed=SEED)

print(f'\n✓ Mixed dataset created: {len(mixed_train):,} examples')
print('✓ No overlap between clean and adversarial sources')

## 6. Prepare Val/Test Sets

In [ ]:
val_ds = clean_ds['validation'].remove_columns(
    [c for c in clean_ds['validation'].column_names if c not in keep_cols]
)
test_ds = clean_ds['test'].remove_columns(
    [c for c in clean_ds['test'].column_names if c not in keep_cols]
)
print(f'Val: {len(val_ds)}, Test: {len(test_ds)}')

## 7. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL, use_fast=True)

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

tokenized_train = mixed_train.map(tok, batched=True)
tokenized_val = val_ds.map(tok, batched=True)
tokenized_test = test_ds.map(tok, batched=True)

for ds in [tokenized_train, tokenized_val, tokenized_test]:
    if 'text' in ds.column_names:
        ds = ds.remove_columns(['text'])

tokenized_train = tokenized_train.rename_column('label_id', 'labels')
tokenized_val = tokenized_val.rename_column('label_id', 'labels')
tokenized_test = tokenized_test.rename_column('label_id', 'labels')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenization complete')

## 8. Class Weights

In [ ]:
train_labels = np.array(tokenized_train['labels'])
counts = np.bincount(train_labels, minlength=len(LABELS)).astype(np.float32)
freq = counts / counts.sum()
inv = 1.0 / (freq + 1e-9)
weights = inv / inv.mean()
class_weights = torch.tensor(weights, dtype=torch.float32)

print('Class weights:')
for i, (l, w) in enumerate(zip(LABELS, weights)):
    print(f'  {l}: {w:.3f}')

## 9. Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASELINE_MODEL,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} params')

## 10. Custom Trainer

In [ ]:
LABEL_SMOOTHING = 0.05

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**{k:v for k,v in inputs.items() if k != 'labels'})
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            label_smoothing=LABEL_SMOOTHING
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
    }

## 11. Training Configuration

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    
    num_train_epochs=4,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
    seed=42,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)

train_len = len(tokenized_train)
steps_per_epoch = math.ceil(train_len / (args.per_device_train_batch_size * args.gradient_accumulation_steps))
num_training_steps = steps_per_epoch * int(args.num_train_epochs)
num_warmup_steps = int(args.warmup_ratio * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f'Training steps: {num_training_steps}, Warmup: {num_warmup_steps}')

## 12. Train

In [ ]:
trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    optimizers=(optimizer, scheduler),
)

print('\n' + '='*70)
print('STARTING ADVERSARIAL TRAINING (NON-OVERLAPPING DATA)')
print('='*70)
print(f'Clean examples (non-overlapping): {n_clean_use:,}')
print(f'Adversarial examples: {n_adv_use:,}')
print(f'Total training examples: {len(tokenized_train):,}')
print('='*70)

train_result = trainer.train()
print(f'\nTraining complete! Loss: {train_result.training_loss:.4f}')

## 13. Save Model

In [ ]:
final_dir = os.path.join(OUTPUT_DIR, 'final_model')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'Model saved to {final_dir}')

## 14. Evaluate

In [ ]:
val_results = trainer.evaluate(tokenized_val)
test_results = trainer.evaluate(tokenized_test)

print('\nValidation Results:')
print(f"  Accuracy: {val_results['eval_accuracy']:.4f}")
print(f"  F1 Macro: {val_results['eval_f1_macro']:.4f}")

print('\nTest Results:')
print(f"  Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"  F1 Macro: {test_results['eval_f1_macro']:.4f}")

pred = trainer.predict(tokenized_test)
preds = np.argmax(pred.predictions, axis=-1)
print('\nClassification Report:')
print(classification_report(pred.label_ids, preds, target_names=LABELS, digits=4))

## 15. Save Results with Overlap Info

In [ ]:
results = {
    'training_data': {
        'clean_total': n_clean_total,
        'attacked_examples': n_attacked,
        'clean_available': n_clean_available,
        'clean_used': n_clean_use,
        'adversarial_used': n_adv_use,
        'total_training': n_clean_use + n_adv_use,
        'overlap': 0,
        'adv_ratio': ADVERSARIAL_RATIO,
        'actual_adv_ratio': n_adv_use / (n_clean_use + n_adv_use)
    },
    'test_results': {
        'accuracy': float(test_results['eval_accuracy']),
        'f1_macro': float(test_results['eval_f1_macro']),
        'f1_weighted': float(test_results['eval_f1_weighted'])
    }
}

with open(os.path.join(OUTPUT_DIR, 'results_nonoverlap.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('\n' + '='*70)
print('FINAL SUMMARY')
print('='*70)
print(f'Original clean data: {n_clean_total:,}')
print(f'Excluded (used for adv generation): {n_attacked:,}')
print(f'Clean data used for training: {n_clean_use:,}')
print(f'Adversarial data used: {n_adv_use:,}')
print(f'Total training examples: {n_clean_use + n_adv_use:,}')
print(f'Overlap: 0 examples ✓')
print('='*70)
print('Results saved!')